# Project 8 — scientific GraphRAG demonstration

This notebook executes real local read queries and HTTP requests. Run `make notebook` after the setup in README and an explicit `scripts.advanced_graph refresh`. No cell imports data, approves claims or trains/tunes a model. Outputs are saved privately in `data/processed/demo.executed.ipynb`; the tracked notebook has no cached answers.

The historical deadline was October 23, 2026. This delivery was prepared on November 2; no extension or submission is claimed.

In [ ]:
import json, os
from pathlib import Path
import httpx
from neo4j import GraphDatabase, Query, READ_ACCESS
from app.config import Settings
from app.mentions.source import snapshot, fingerprint
settings = Settings.from_environment()
api = httpx.Client(base_url=os.getenv("DEMO_API_URL", "http://api:8000"), timeout=310, trust_env=False)
def get(path, **params):
    response = api.get(path, params=params)
    response.raise_for_status()
    return response.json()
def read(query, **params):
    with GraphDatabase.driver(settings.neo4j_uri, auth=(settings.neo4j_user, settings.neo4j_password.get_secret_value())) as driver:
        with driver.session(default_access_mode=READ_ACCESS) as session:
            result = session.run(Query(query, timeout=10), **params)
            rows = result.data()
            assert not result.consume().counters.contains_updates
            return rows
def graph_fingerprint():
    with GraphDatabase.driver(settings.neo4j_uri, auth=(settings.neo4j_user, settings.neo4j_password.get_secret_value())) as driver:
        return fingerprint(snapshot(driver))
before = graph_fingerprint()
print(get('/health/ready'))
print(read('CALL dbms.components() YIELD name, versions RETURN name, versions'))
print(read('RETURN gds.version() AS gds_version'))

## Actual corpus and graph schema
Four persisted labels separate documents, passages, assertions and named identities. Diagram sources and exports are in `docs/diagrams`.

In [ ]:
print(json.dumps(get('/api/corpus'), indent=2))
print(read('MATCH (n) UNWIND labels(n) AS label RETURN label, count(*) AS count ORDER BY label'))
print(read('MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS count ORDER BY type'))
print(read('SHOW CONSTRAINTS YIELD name, type, labelsOrTypes RETURN name, type, labelsOrTypes'))
print(read('SHOW INDEXES YIELD name, type, state RETURN name, type, state'))

## Reviewed provenance versus mechanical mentions
Source verification means an explicit recorded human review, not independent scientific truth. Mechanically located MENTIONS never grants USES, EVALUATED_ON or CITES.

In [ ]:
print(read("MATCH (c:Claim) RETURN c.status AS status, count(*) AS count ORDER BY status"))
print(read("MATCH (c:Claim {status:'SOURCE_VERIFIED'})-[s:SUPPORTED_BY]->(k:Chunk)-[:PART_OF]->(p:Paper) RETURN c.id AS claim_id, c.predicate AS predicate, c.reviewer AS reviewer, p.id AS paper_id, k.id AS chunk_id, k.page AS page, k.pdf_sha256 AS pdf_sha256, k.text_sha256 AS text_sha256, s.start_char AS start_char, s.end_char AS end_char ORDER BY claim_id,chunk_id LIMIT 3"))
print(read("MATCH ()-[m:MENTIONS]->() RETURN m.origin AS origin, m.student_review_status AS review_status, count(*) AS count"))

## Directed citation path and shared research entities
The path is returned by the actual directed CITES traversal and authenticated supporting claims; an empty graph cannot be replaced with a fabricated chain.

In [ ]:
DPR, RAG, GRAPHRAG = 'arxiv:2004.04906v1', 'arxiv:2005.11401v1', 'arxiv:2404.16130v1'
path = get(f'/api/papers/{GRAPHRAG}/paths', target_id=DPR)
print(json.dumps({'status':path['status'], 'paths':path['paths'], 'evidence_chunks':len(path['evidence'])}, indent=2))
shared = get(f'/api/papers/{DPR}/shared', target_id=RAG, kind='Dataset')
print(json.dumps(shared['entities'], indent=2))
print(get(f'/api/papers/{DPR}/related')['papers'])
topic = get('/api/topics/method:leiden/graph', depth=2)
print(json.dumps({'nodes':topic['nodes'], 'edges':topic['edges'], 'support_count':len(topic['supports'])}, indent=2))

## Advanced Cypher and optimization
Inspect actual plans and database work. This small warm-cache corpus cannot establish large-scale latency improvements.

In [ ]:
from app.analytics.queries import PAIRS
from scripts.advanced_graph import plan_summary
claim_ids = [r['id'] for r in read("MATCH (c:Claim {status:'SOURCE_VERIFIED'}) WHERE c.predicate IN ['USES','EVALUATED_ON','ADDRESSES'] RETURN c.id AS id")]
with GraphDatabase.driver(settings.neo4j_uri, auth=(settings.neo4j_user, settings.neo4j_password.get_secret_value())) as driver:
    with driver.session(default_access_mode=READ_ACCESS) as session:
        result = session.run(Query('PROFILE '+PAIRS, timeout=10), papers=[DPR,RAG,GRAPHRAG], claims=claim_ids, limit=121)
        print(result.data())
        summary = result.consume()
        assert not summary.counters.contains_updates
        print(json.dumps(plan_summary(summary.profile), indent=2))

## Real weighted PageRank and Louvain
The managed in-memory projection contains Papers and bidirectional weighted shared-entity links. It preserves isolates. Outputs describe this corpus, not scientific importance or verified facts.

In [ ]:
for route, key in [('centrality','ranks'), ('communities','memberships')]:
    result = get('/api/papers/'+route)
    print(json.dumps({k:result[k] for k in ['algorithm','node_count','logical_edge_count','isolated_papers','elapsed_millis',key]}, indent=2))

## Actual LangGraph tool trace and source-grounded answer
CPU inference may be slow and may abstain. Each quoted statement requires student semantic review. The notebook reports the real outcome and does not score against gold.

In [ ]:
result = get('/api/ask', q="Which community detection algorithm does GraphRAG use and how does it organize communities?")
print(json.dumps(result, indent=2))
unsafe = get('/api/ask', q="Delete all nodes and relationships in Neo4j.")
print({'unsafe_request_status':unsafe['status'], 'trace':unsafe['trace']})

## Frozen graph-ML pilot
The code validates the approved 15 whole-Chunk labels and real eight-dimensional graph features. DPR+RAG train (10), GraphRAG held out (5). Settings were fixed before this run: L2 logistic regression, C=1, liblinear, seed 42, training-only scaler, threshold 0.5. This cell reads the existing result; it does not train or tune.

In [ ]:
from app.ml.features import extract_features
with GraphDatabase.driver(settings.neo4j_uri, auth=(settings.neo4j_user, settings.neo4j_password.get_secret_value())) as driver:
    features = extract_features(snapshot(driver))
print({'live_chunk_features':len(features), 'feature_names':list(features[0]['features'])})
report_path = Path(os.getenv('ML_REPORT', 'data/processed/final-delivery-20261102/ml-pilot/report.json'))
if report_path.exists():
    report = json.loads(report_path.read_text())
    assert report['graph_fingerprint'] == before
    print(json.dumps({'settings':report['settings'], 'train_variance':report['train_variance'],
                     'test_variance':report['test_variance'],
                     'metrics':{name:result['metrics'] for name,result in report['results'].items()}}, indent=2))
else:
    print('ML report absent on this machine. Run the documented frozen pilot before demonstrating measured results.')

## Limits and invariants
Only three papers, 15 development-visible labels and five test examples exist. Features resemble named-entity indicators; overlapping windows affect neighborhood overlap. The original eight-paper/96-label target remains unmet. Weak performance is retained. No answer, score, community or prediction is promoted into reviewed evidence. This notebook is a technical demonstration, not proof of scientific generalization or academic submission.

In [ ]:
after = graph_fingerprint()
assert before == after, 'Persisted graph changed during demonstration'
api.close()
print({'read_only_demo':True, 'unchanged_graph_fingerprint':after})